In [ ]:
%pip install -q langgraph

In [ ]:
# Dana's Bakery evaluates each order against the same stock snapshot.
from pprint import pprint
from typing import Literal, TypedDict

from IPython.display import Image, display
from langgraph.graph import END, START, StateGraph


class OrderInput(TypedDict):
    item: str
    quantity: int
    day: str
    city: str | None


class OrderResult(TypedDict):
    status: Literal["accepted", "rejected"]
    item: str | None
    delivery: Literal["pickup", "delivery"]
    total: float
    message: str
    steps: list[str]


class OrderState(OrderResult):
    quantity: int
    day: str
    city: str
    requested_item: str
    candidate: str
    attempts: int
    in_stock: bool
    rejection_reason: str


# Values are (unit price in shekels, starting stock).
MENU = {
    "challah": (18, 10),
    "babka": (45, 0),
    "rugelach": (30, 3),
    "sourdough": (25, 4),
}
SUBSTITUTES = {"babka": "rugelach", "rugelach": "sourdough", "sourdough": "challah"}
WEEKDAYS = {"sunday", "monday", "tuesday", "wednesday", "thursday", "friday", "saturday"}
MAX_SUBSTITUTES = 2
DELIVERY_FEE = 20

In [ ]:
def validate(state: OrderState) -> dict:
    item = state.get("item")
    quantity = state.get("quantity")
    day = state.get("day")
    city = state.get("city")

    invalid_order = {
        "steps": ["validate"],
        "rejection_reason": (
            "Sorry, we couldn't understand your order. Please choose a current menu item, "
            "enter a positive whole quantity, and include a valid day."
        ),
    }
    if not isinstance(item, str) or not isinstance(day, str):
        return invalid_order
    if city is not None and not isinstance(city, str):
        return invalid_order

    item = item.strip().casefold()
    day = day.strip().casefold()
    if item not in MENU or type(quantity) is not int or quantity <= 0 or day not in WEEKDAYS:
        return invalid_order

    return {
        "steps": ["validate"],
        "requested_item": item,
        "candidate": item,
        "day": day,
        "city": (city or "").strip().casefold(),
        "attempts": 0,
        "rejection_reason": "",
    }


def route_after_validation(state: OrderState) -> Literal["check_stock", "reject"]:
    return "reject" if state["rejection_reason"] else "check_stock"


def check_stock(state: OrderState) -> dict:
    candidate = state["candidate"]
    quantity = state["quantity"]
    available = MENU[candidate][1]
    in_stock = available >= quantity
    outcome = "enough" if in_stock else "short"
    return {
        "in_stock": in_stock,
        "steps": [*state["steps"], f"check_stock ({candidate}): {available} available, {quantity} needed ({outcome})"],
    }


def route_after_stock_check(state: OrderState) -> Literal["price", "substitute", "reject"]:
    if state["in_stock"]:
        return "price"
    if state["attempts"] < MAX_SUBSTITUTES and state["candidate"] in SUBSTITUTES:
        return "substitute"
    return "reject"


def substitute(state: OrderState) -> dict:
    candidate = SUBSTITUTES[state["candidate"]]
    attempt = state["attempts"] + 1
    return {
        "candidate": candidate,
        "attempts": attempt,
        "steps": [*state["steps"], f"substitute ({attempt}): {candidate}"],
    }


def goods_total(subtotal: int, day: str) -> float:
    if subtotal > 150 and day != "friday":
        return round(subtotal * 85 / 100, 2)
    return float(subtotal)


def price(state: OrderState) -> dict:
    unit_price = MENU[state["candidate"]][0]
    subtotal = unit_price * state["quantity"]
    return {
        "total": goods_total(subtotal, state["day"]),
        "steps": [*state["steps"], "price"],
    }


def fulfillment(state: OrderState) -> dict:
    delivery = "delivery" if state["city"] == "jerusalem" else "pickup"
    fee = DELIVERY_FEE if delivery == "delivery" else 0
    return {
        "delivery": delivery,
        "total": round(state["total"] + fee, 2),
        "steps": [*state["steps"], "fulfillment"],
    }


def accept(state: OrderState) -> dict:
    item = state["candidate"]
    quantity = state["quantity"]
    product = f"{quantity} × {item}"
    if item == "rugelach":
        boxes = "box" if quantity == 1 else "boxes"
        product = f"{quantity} {boxes} of rugelach"
    message = f"Thanks for ordering from Dana's Bakery! We've set aside {product} for you."
    if item != state["requested_item"]:
        message += f" We were short on {state['requested_item']}, so we switched your order to {item}."

    if state["delivery"] == "delivery":
        message += " We'll deliver in Jerusalem. The total includes the ₪20 delivery fee."
    elif state["city"]:
        message += " We deliver only in Jerusalem, so we've kept your order for free pickup."
    else:
        message += " Your order is set for free pickup."

    return {
        "status": "accepted",
        "item": item,
        "message": f"{message} Your total is ₪{state['total']:.2f}.",
        "steps": [*state["steps"], "accept"],
    }


def reject(state: OrderState) -> dict:
    message = state["rejection_reason"]
    if not message:
        message = (
            f"Sorry, we couldn't fill your order for {state['quantity']} × "
            f"{state['requested_item']}. We've cancelled it and hope to bake for you next time."
        )
    return {
        "status": "rejected",
        "item": None,
        "delivery": "pickup",
        "total": 0.0,
        "message": message,
        "steps": [*state["steps"], "reject"],
    }

In [ ]:
builder = StateGraph(OrderState, input_schema=OrderInput, output_schema=OrderResult)

builder.add_node("validate", validate)
builder.add_node("check_stock", check_stock)
builder.add_node("substitute", substitute)
builder.add_node("price", price)
builder.add_node("fulfillment", fulfillment)
builder.add_node("accept", accept)
builder.add_node("reject", reject)

builder.add_edge(START, "validate")
builder.add_conditional_edges("validate", route_after_validation)
builder.add_conditional_edges("check_stock", route_after_stock_check)
builder.add_edge("substitute", "check_stock")
builder.add_edge("price", "fulfillment")
builder.add_edge("fulfillment", "accept")
builder.add_edge("accept", END)
builder.add_edge("reject", END)

graph = builder.compile()

In [ ]:
# Mermaid rendering uses an online renderer and needs no API key.
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
# Expected results include the stock checks in their actual execution order.
accepted_tail = ["price", "fulfillment", "accept"]
examples = [
    (
        "Direct pickup with weekday discount",
        {"item": " ChAllAh ", "quantity": 10, "day": " THURSDAY ", "city": ""},
        ("accepted", "challah", "pickup", 153),
        ["validate", "check_stock (challah): 10 available, 10 needed (enough)", *accepted_tail],
        "free pickup",
    ),
    (
        "First substitute with Jerusalem delivery",
        {"item": "babka", "quantity": 2, "day": "Thursday", "city": " jerusalem "},
        ("accepted", "rugelach", "delivery", 80),
        [
            "validate",
            "check_stock (babka): 0 available, 2 needed (short)",
            "substitute (1): rugelach",
            "check_stock (rugelach): 3 available, 2 needed (enough)",
            *accepted_tail,
        ],
        "switched your order to rugelach",
    ),
    (
        "Second substitute with outside-city pickup",
        {"item": "babka", "quantity": 4, "day": "Thursday", "city": "Tel Aviv"},
        ("accepted", "sourdough", "pickup", 100),
        [
            "validate",
            "check_stock (babka): 0 available, 4 needed (short)",
            "substitute (1): rugelach",
            "check_stock (rugelach): 3 available, 4 needed (short)",
            "substitute (2): sourdough",
            "check_stock (sourdough): 4 available, 4 needed (enough)",
            *accepted_tail,
        ],
        "We deliver only in Jerusalem",
    ),
    (
        "Cancellation at the two-substitute limit",
        {"item": "babka", "quantity": 5, "day": "Thursday", "city": ""},
        ("rejected", None, "pickup", 0),
        [
            "validate",
            "check_stock (babka): 0 available, 5 needed (short)",
            "substitute (1): rugelach",
            "check_stock (rugelach): 3 available, 5 needed (short)",
            "substitute (2): sourdough",
            "check_stock (sourdough): 4 available, 5 needed (short)",
            "reject",
        ],
        "cancelled",
    ),
    (
        "Cancellation at the end of the chain",
        {"item": "sourdough", "quantity": 11, "day": "Thursday", "city": ""},
        ("rejected", None, "pickup", 0),
        [
            "validate",
            "check_stock (sourdough): 4 available, 11 needed (short)",
            "substitute (1): challah",
            "check_stock (challah): 10 available, 11 needed (short)",
            "reject",
        ],
        "cancelled",
    ),
    (
        "Friday pickup at full price",
        {"item": "challah", "quantity": 10, "day": "Friday", "city": ""},
        ("accepted", "challah", "pickup", 180),
        ["validate", "check_stock (challah): 10 available, 10 needed (enough)", *accepted_tail],
        "₪180.00",
    ),
    (
        "Weekday discount with the full delivery fee",
        {"item": "challah", "quantity": 10, "day": "Thursday", "city": "Jerusalem"},
        ("accepted", "challah", "delivery", 173),
        ["validate", "check_stock (challah): 10 available, 10 needed (enough)", *accepted_tail],
        "₪173.00",
    ),
]

invalid_orders = [
    ("Unknown product", "pizza", 1),
    ("Discontinued product", "croissants", 1),
    ("Zero quantity", "challah", 0),
    ("Negative quantity", "babka", -2),
    ("Fractional quantity", "challah", 1.5),
    ("Boolean quantity", "challah", True),
]
for label, item, quantity in invalid_orders:
    examples.append((
        label,
        {"item": item, "quantity": quantity, "day": "Thursday", "city": ""},
        ("rejected", None, "pickup", 0),
        ["validate", "reject"],
        "couldn't understand",
    ))

result_fields = ("status", "item", "delivery", "total")
for label, order, expected, expected_steps, reply_text in examples:
    result = graph.invoke(order)
    assert set(result) == {*result_fields, "message", "steps"}, label
    assert tuple(result[field] for field in result_fields) == expected, (label, result)
    assert result["steps"] == expected_steps, (label, result["steps"])
    assert reply_text in result["message"], (label, result["message"])
    print(label)
    pprint(result, sort_dicts=False)
    print()

# Five boxes cost exactly ₪150, although the stock snapshot has only three boxes.
assert goods_total(5 * MENU["rugelach"][0], "thursday") == 150
print(f"Passed all {len(examples)} orders and the exact-threshold check.")